In [1]:
%cd ../

/Users/hoangle/Projects/untangling-people/fwo_models


In [2]:
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

# Load meal names

In [3]:
raw_meals = pd.read_parquet("data/processed/phase_4/dim_meals.parquet")
raw_meals.head()

,meal_id,meal_type,schoolyear,restaurant,attributes,aliases
0,9017,vegan,24-25,"[che, exa, vik]","[vegan-miscellaneous, kela]","[""Butter"" härkäpapua & pähkinää]"
1,7201,vegan,23-24,None,[],[2023 Härkäpu-sienilasagnette]
2,9032,vegan,23-24,None,[],[Appelisiini-luomukikhernecurrya]
3,9102,vegan,23-24,None,[],[Artisokkavugetteja & tuoretomaattisalsaa]
4,7010,vegetarian,24-25,"[che, exa, vik]",[],"[Aurajuusto-pinaattilasagnette, Aurajuusto-pin..."


In [4]:
name_list = [
    {
        'meal_id': row.meal_id,
        'name': row.aliases[0]
    }
    for row in raw_meals.itertuples()
]
names = pd.DataFrame.from_records(name_list)
names.head()

,meal_id,name
0,9017,"""Butter"" härkäpapua & pähkinää"
1,7201,2023 Härkäpu-sienilasagnette
2,9032,Appelisiini-luomukikhernecurrya
3,9102,Artisokkavugetteja & tuoretomaattisalsaa
4,7010,Aurajuusto-pinaattilasagnette


# Encode

In [5]:
device = "mps"
tqdm.pandas()


EMBEDDING_MODEL_NAME = "jinaai/jina-embeddings-v3"
tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
model     = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME, trust_remote_code=True, from_tf=False, use_flash_attn=False).to(device)

In [6]:
outputs = model.encode(names['name'].tolist(), task="text-matching")

In [8]:
# embds = np.load("data/inter/meal_names.npz.npy")
names['embedding'] = outputs.tolist()
names.head()

,meal_id,name,embedding
0,9017,"""Butter"" härkäpapua & pähkinää","[0.009897640906274319, -0.026521963998675346, ..."
1,7201,2023 Härkäpu-sienilasagnette,"[0.05687922611832619, -0.0033640582114458084, ..."
2,9032,Appelisiini-luomukikhernecurrya,"[-0.0010028686374425888, 0.0245476383715868, 0..."
3,9102,Artisokkavugetteja & tuoretomaattisalsaa,"[0.0934152752161026, 0.032747749239206314, 0.1..."
4,7010,Aurajuusto-pinaattilasagnette,"[0.011745608411729336, -0.03860291838645935, 0..."


In [11]:
path = "data/inter/meal_names_embds.parquet"
names.to_parquet(path)